In [ ]:
import os

# 1. Le decimos al compilador dónde está CUDA y qué GPU exacta estamos usando (T4 = arch 7.5)
os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"

# 2. Instalamos las herramientas de construcción faltantes en Python 3.12
!pip install -q wheel setuptools ninja

# 3. Reintentamos la compilación (esto tomará unos 5 minutos)
print("⏳ Recompilando diff-gaussian-rasterization con Ninja...")
!pip install ./submodules/diff-gaussian-rasterization

print("⏳ Recompilando simple-knn con Ninja...")
!pip install ./submodules/simple-knn

print("¡Módulos instalados correctamente!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 15.6 MB/s eta 0:00:00
⏳ Recompilando diff-gaussian-rasterization con Ninja...
Processing ./submodules/diff-gaussian-rasterization
  Preparing metadata (setup.py) ... done
  Created wheel for diff_gaussian_rasterization: filename=diff_gaussian_rasterization-0.0.0-cp312-cp312-linux_x86_64.whl size=3584996 sha256=cfe89fc078665aade896b9033295f334ba902223fe84f133f8e4105105febdb7
  Stored in directory: /root/.cache/pip/wheels/01/e0/e8/f40a1cd6a1d5760cbd3036081bdad5b36c41fc11c786d4a404
Successfully built diff_gaussian_rasterization
  Attempting uninstall: diff_gaussian_rasterization
    Found existing installation: diff_gaussian_rasterization 0.0.0
    Uninstalling diff_gaussian_rasterization-0.0.0:
      Successfully uninstalled diff_gaussian_rasterization-0.0.0
⏳ Recompilando simple-knn con Ninja...
Processing ./submodules/simple-knn
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.p

In [ ]:
import os

# 1. Limpiamos los archivos temporales corruptos del intento anterior
!rm -rf ./submodules/simple-knn/build
!rm -rf ./submodules/simple-knn/*.egg-info

# 2. Forzamos a que compile usando un solo núcleo para no saturar la RAM
os.environ["MAX_JOBS"] = "1"

# 3. Instalamos mostrando todos los detalles (modo verboso)
print("⏳ Compilando simple-knn de forma segura (MAX_JOBS=1)...")
!pip install -v ./submodules/simple-knn

print("✅ ¡Proceso terminado!")

⏳ Compilando simple-knn de forma segura (MAX_JOBS=1)...
Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
Processing ./submodules/simple-knn
  Running command python setup.py egg_info
  running egg_info
  creating /tmp/pip-pip-egg-info-kylhto4f/simple_knn.egg-info
  writing /tmp/pip-pip-egg-info-kylhto4f/simple_knn.egg-info/PKG-INFO
  writing dependency_links to /tmp/pip-pip-egg-info-kylhto4f/simple_knn.egg-info/dependency_links.txt
  writing top-level names to /tmp/pip-pip-egg-info-kylhto4f/simple_knn.egg-info/top_level.txt
  writing manifest file '/tmp/pip-pip-egg-info-kylhto4f/simple_knn.egg-info/SOURCES.txt'
  reading manifest file '/tmp/pip-pip-egg-info-kylhto4f/simple_knn.egg-info/SOURCES.txt'
  writing manifest file '/tmp/pip-pip-egg-info-kylhto4f/simple_knn.egg-info/SOURCES.txt'
  Preparing metadata (setup.py) ... done
  Running command python setup.py bdist_wheel
  running bdist_wheel
  running build
  running build_ext
  building 'simple_knn._C' 

In [ ]:
# 1. Inyectamos la librería faltante en la primera línea del archivo problemático
!sed -i '1i #include <float.h>' ./submodules/simple-knn/simple_knn.cu

# 2. Limpiamos la basura del intento fallido anterior
!rm -rf ./submodules/simple-knn/build
!rm -rf ./submodules/simple-knn/*.egg-info

# 3. Instalamos nuevamente
print("⏳ Compilando simple-knn con el parche de float.h aplicado...")
!pip install -q ./submodules/simple-knn

print("✅ ¡Proceso terminado con éxito! Ya puedes continuar.")

⏳ Compilando simple-knn con el parche de float.h aplicado...
  Preparing metadata (setup.py) ... done
✅ ¡Proceso terminado con éxito! Ya puedes continuar.


In [ ]:
import os

# Crear carpeta de datos y descargar
os.makedirs("data", exist_ok=True)
!wget -O tandem.zip https://huggingface.co/camenduru/gaussian-splatting/resolve/main/tandem.zip
!unzip -q tandem.zip -d data/tandem

print("Dataset 'Tandem' descargado y descomprimido en data/tandem.")

--2026-06-09 04:43:04--  https://huggingface.co/camenduru/gaussian-splatting/resolve/main/tandem.zip
Resolving huggingface.co (huggingface.co)... 13.35.202.121, 13.35.202.40, 13.35.202.34, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.121|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-06-09 04:43:04 ERROR 404: Not Found.

[tandem.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of tandem.zip or
        tandem.zip.zip, and cannot find tandem.zip.ZIP, period.
Dataset 'Tandem' descargado y descomprimido en data/tandem.


In [ ]:
import os
import shutil
import glob

# 1. Limpiar intentos anteriores para evitar conflictos
if os.path.exists("data/tandem"):
    shutil.rmtree("data/tandem")

# 2. Descomprimir el archivo zip que está en la raíz
print("📦 Descomprimiendo tandem.zip...")
!unzip -q tandem.zip -d data/tandem

# 3. Buscar automáticamente dónde quedó la carpeta de cámaras ('sparse/0')
rutas_sparse = glob.glob("data/tandem/**/sparse/0", recursive=True)

if rutas_sparse:
    # Obtener la carpeta principal (dos niveles arriba de sparse/0)
    ruta_correcta = os.path.dirname(os.path.dirname(rutas_sparse[0]))
    print(f"✅ ¡Ruta detectada con éxito! La escena está en: {ruta_correcta}")
    print(f"🚀 Iniciando entrenamiento...\n")

    # Lanzar el entrenamiento con la ruta detectada
    !python train.py -s {ruta_correcta} --iterations 2000
else:
    print("❌ No se encontró la carpeta 'sparse' dentro del zip. Verificando el contenido extraído:")
    !ls -la data/tandem/

📦 Descomprimiendo tandem.zip...
[tandem.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of tandem.zip or
        tandem.zip.zip, and cannot find tandem.zip.ZIP, period.
❌ No se encontró la carpeta 'sparse' dentro del zip. Verificando el contenido extraído:
ls: cannot access 'data/tandem/': No such file or directory
